# Ultra Small Language Models benchmarked

## Ollama Installation alongwith ZStandard

In [ ]:
!nvidia-smi

Thu May 21 11:00:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!apt install

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.


In [ ]:
!apt-get update && apt install zstd -y
!pip install ollama
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [77.8 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,945 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,070 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/ma

## Setup and Begin Running Ollama Server

In [ ]:
import os
import threading
import subprocess
import ollama

def run_ollama():
    os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
    subprocess.Popen(["ollama", "serve"])

threading.Thread(target=run_ollama).start()


### Decorator to log function calls

In [ ]:
# Decorator function to help in logging the tools called by the Models

import functools

logs = {}

log_count = 0

def logCount_Incrementor(log_count):
    """Increments the Log Count by 1"""
    log_count += 1
    return log_count

#Tool Use tracker through decorators
def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        #Log before function runs
        logs.update({logCount_Incrementor(log_count) :f"Log Calling {func.__name__} with args: f{args} and  kwargs:{kwargs}"})

        return func(*args, **kwargs)

    return wrapper

In [ ]:
import json
import random

#Get Machine State
@log_call
def get_machine_state(**kwargs)-> str:
    """Get Current Machine State"""
    return json.dumps({
        'State':'Active'
    })

#Get Weather
@log_call
def get_weather(location: str, **kwargs) ->str:
    """Get the weather based on the location provided by the user"""
    return json.dumps({
        'location': location,
        'temperature': 36
    })

#Validate Switch
@log_call
def validate_switch(**kwargs):
    """Check the status of the switch"""
    return json.dumps({
        "Status":"Switch Status: ON"
        })

#Setup Meeting
meetings = {}
@log_call
def setup_meeting(persons: int, time: str, **kwargs):
    """Schedule a meeting"""
    meetings.update({'Time': time, "Persons":persons})
    return json.dumps({
        'message':f"Meeting has been scheduled as requested: Persons: {persons} Time: {time}"
        })

#Get Population
@log_call
def get_population(location:str, **kwargs):
    """Get population data"""
    return json.dumps({
        'population':100000
    })

# Available tools
available_tools = {
        'get_machine_state' :   get_machine_state,
        'get_weather'       :   get_weather,
        'validate_switch'   :   validate_switch,
        'setup_meeting'     :   setup_meeting,
        'get_population'    :   get_population,
}

===========================================================================================================


# **Actual Benchmarking**

In [ ]:
ROLE = 'user'

#Experimental Prompts
get_machine_state
get_weather
validate_switch
setup_meeting
get_population

P1  = "What is the Current Machine state?"
P2  = "What is the weather like in Mauritius?"
P3  = "Have I turned off the Switch?"
P4  = "Do schedule a meeting for 8 persons at 12 PM."
P5  = "My Machine state is Active, what's yours?"
P6  = "Whats the population of London?"
P7  = "Dont check the machine state, just tell me if the switch is ON?"
P8  = "Blue Ocean Meeting Bits Dublin"
P9  = "I do not want to schedule a meeting of 2 persons at 6pm."
P10 = "The population of India is 150000000000. Whats the weather like in Delhi?"
P11 = "List the tools you have access to."
P12 = "I want to visit Kolkata, anything I should know?"
P13 = "How could he have set up a meeting for 6 people at 4pm without notifying me?"
P14 = "Tommorrow is a holiday, I shouldn't set up a meeting, right?"
P15 = "Why is population so hard to calculate?"
P16 = "If I ask you to schedule a meeting for 8 people at 3pm, which tool would u use?"
P17 = "If the weather is windy in California, how is it in Madagascar?"
P18 = "Tell me about the machine state, but only if the switch is off."
P19 = "Can you get the weather for both London and Tokyo?"
P20 = "I want to know the population of Tokyo and the machine state."
P21 = "Schedule a meeting for 3 at 5pm and then validate the switch."
P22 = "What's the weather in Paris, and what's the population of France?"
P23 = "If the machine is active, tell me the weather in New York."
P24 = "Is the switch on, and if so, what's the machine state?"
P25 = "I need to know the population of China, and also schedule a meeting for 10 people at 9 AM."
P26 = "What's the weather in Sydney right now, and also, is the machine active?"
P27 = "I need to schedule a meeting for myself and two others at 7 PM. Also, what's the population of Germany?"
P28 = "Can you tell me the weather in Rome and check the switch status?"
P29 = "What is the current machine state, and what's the weather in Berlin?"
P30 = "If I ask for the machine state, but also want the switch to be off, what happens?"

prompts = [P1, P2, P3, P4, P5, P6, P7,P8, P9, P10, P11, P12, P13, P14, P15, P16, P17, P18, P19, P20, P21, P22, P23, P24, P25, P26, P27, P28, P29, P30]

prerequisite = ""

## Message Generator
def messageGen(prompt: str):
    return {'role':ROLE,
            'content': prompt}

### Model 1: qwen2.5:0.5b

In [ ]:
!ollama pull qwen2.5:0.5b

In [ ]:
from ollama import chat
import ollama
import json

for prompt in prompts:
    print(f"\n--- Current Prompt: {prompt} ---")

    messages = [messageGen(prompt)]
    response = chat(
        model='qwen2.5:0.5b',
        messages = messages,
        tools = [get_machine_state, get_weather, validate_switch, setup_meeting, get_population]
    )

    if response.get('message') and response['message'].get('tool_calls'):
        # Add assistant's tool call intent to message history
        messages.append(response['message'])

        for tool_call in response['message']['tool_calls']:
            function_to_call = tool_call['function']['name']
            function_args = tool_call['function']['arguments']

            if function_to_call in available_tools:
                print(f"[Tool Calling]: {function_to_call} with {function_args}")
                try:
                    tool_output = available_tools[function_to_call](**function_args)
                except TypeError as e:
                    print(f"[Error]: Tool call failed for {function_to_call} with arguments {function_args}: {e}")
                    tool_output = json.dumps({"error": str(e), "tool": function_to_call, "args": function_args})

                # Feed the tool result back using the 'content' key
                messages.append({
                    'role': 'tool',
                    'content': tool_output
                })

        # Final call to get the text response based on tool results
        final_response = ollama.chat(model = 'qwen2.5:0.5b', messages = messages)
        print("Assistant Response:", final_response['message']['content'])
    else:
        print("Assistant Response:", response['message']['content'])

print("\nExecution Logs:", logs)


--- Current Prompt: What is the Current Machine state? ---
[Tool Calling]: get_machine_state with {'kwargs': ''}
Assistant Response: The current machine state in Alibaba Cloud is "Active". This indicates that the machine is currently running and ready to receive tasks. If you need any further information or assistance with this, feel free to ask!

--- Current Prompt: What is the weather like in Mauritius? ---
[Tool Calling]: get_weather with {'location': 'Mauritius'}
Assistant Response: Mauritius experiences a temperature of around 36 degrees Celsius. However, it's important to note that the weather can vary from day to day and is influenced by factors like wind patterns, air quality, and historical weather conditions. If you plan on visiting Mauritius soon or have specific expectations for your stay, it would be best to check the current weather forecast before your journey.

--- Current Prompt: Have I turned off the Switch? ---
Assistant Response: To check if you've turned off the s

In [ ]:
print(logs)

{1: 'Log Calling get_machine_state with args: f() and  kwargs:{}'}


### Model2: qwen3: 0.6b

In [ ]:
!ollama pull qwen3:0.6b

In [ ]:
# Initial Test Runs with qwen2.5:0.5b
from ollama import chat
import ollama
import json

for prompt in prompts:
    print(f"\n--- Current Prompt: {prompt} ---")

    messages = [messageGen(prompt)]
    response = chat(
        model='qwen3:0.6b',
        messages = messages,
        tools = [get_machine_state, get_weather, validate_switch, setup_meeting, get_population]
    )

    if response.get('message') and response['message'].get('tool_calls'):
        # Add assistant's tool call intent to message history
        messages.append(response['message'])

        for tool_call in response['message']['tool_calls']:
            function_to_call = tool_call['function']['name']
            function_args = tool_call['function']['arguments']

            if function_to_call in available_tools:
                print(f"[Tool Calling]: {function_to_call} with {function_args}")
                try:
                    tool_output = available_tools[function_to_call](**function_args)
                except TypeError as e:
                    print(f"[Error]: Tool call failed for {function_to_call} with arguments {function_args}: {e}")
                    tool_output = json.dumps({"error": str(e), "tool": function_to_call, "args": function_args})

                # Feed the tool result back using the 'content' key
                messages.append({
                    'role': 'tool',
                    'content': tool_output
                })

        # Final call to get the text response based on tool results
        final_response = ollama.chat(model = 'qwen3:0.6b', messages = messages)
        print("Assistant Response:", final_response['message']['content'])
    else:
        print("Assistant Response:", response['message']['content'])

print("\nExecution Logs:", logs)


--- Current Prompt: What is the Current Machine state? ---
Assistant Response: 

--- Current Prompt: What is the weather like in Mauritius? ---
[Tool Calling]: get_weather with {'location': 'Mauritius'}
Assistant Response: The weather in Mauritius is currently chilly with a temperature of **36°C**. Let me know if you need further updates! 🌪️

--- Current Prompt: Have I turned off the Switch? ---
[Tool Calling]: validate_switch with {'kwargs': 'switch'}
Assistant Response: The Switch is currently **ON**. If you'd like to turn it off, you can do so by pressing the power button. Let me know if you need further assistance!

--- Current Prompt: Do schedule a meeting for 8 persons at 12 PM. ---
[Tool Calling]: setup_meeting with {'time': '12 PM', 'persons': 8}
Assistant Response: The meeting has been scheduled as requested: **Persons: 8 Time: 12 PM**. Let me know if you need further assistance!

--- Current Prompt: My Machine state is Active, what's yours? ---
[Tool Calling]: get_machine_st

### Model3: qwen3:4b

In [ ]:
!ollama pull qwen3:4b

In [ ]:
# Initial Test Runs with qwen3:4b
from ollama import chat
import ollama
import json

for prompt in prompts:
    print(f"\n--- Current Prompt: {prompt} ---")

    messages = [messageGen(prompt)]
    response = chat(
        model='qwen3:4b',
        messages = messages,
        tools = [get_machine_state, get_weather, validate_switch, setup_meeting, get_population]
    )

    if response.get('message') and response['message'].get('tool_calls'):
        # Add assistant's tool call intent to message history
        messages.append(response['message'])

        for tool_call in response['message']['tool_calls']:
            function_to_call = tool_call['function']['name']
            function_args = tool_call['function']['arguments']

            if function_to_call in available_tools:
                print(f"[Tool Calling]: {function_to_call} with {function_args}")
                try:
                    tool_output = available_tools[function_to_call](**function_args)
                except TypeError as e:
                    print(f"[Error]: Tool call failed for {function_to_call} with arguments {function_args}: {e}")
                    tool_output = json.dumps({"error": str(e), "tool": function_to_call, "args": function_args})

                # Feed the tool result back using the 'content' key
                messages.append({
                    'role': 'tool',
                    'content': tool_output
                })

        # Final call to get the text response based on tool results
        final_response = ollama.chat(model = 'qwen3:4b', messages = messages)
        print("Assistant Response:", final_response['message']['content'])
    else:
        print("Assistant Response:", response['message']['content'])

print("\nExecution Logs:", logs)


--- Current Prompt: What is the Current Machine state? ---
[Tool Calling]: get_machine_state with {'type': 'current'}
Assistant Response: The current machine state is **Active**.

--- Current Prompt: What is the weather like in Mauritius? ---
[Tool Calling]: get_weather with {'location': 'Mauritius'}
Assistant Response: The current temperature in **Mauritius** is **36°C** (as of the latest data).  

Mauritius has a **tropical climate** with warm, humid conditions year-round. Expect consistent temperatures between **25°C–32°C** (77°F–90°F), with high humidity and occasional rain showers. The weather is generally pleasant for travel, especially during the dry season (April–October), though the island experiences a short rainy season (November–March).  

Let me know if you'd like more details! 😊

--- Current Prompt: Have I turned off the Switch? ---
[Tool Calling]: validate_switch with {'kwargs': ''}
Assistant Response: The Switch is currently **ON**. So no, you haven't turned it off. 😊


### Model4: ministral-3:3b

In [ ]:
!ollama pull ministral-3:3b

In [ ]:
# Initial Test Runs with phi4-mini:3.8b
from ollama import chat
import ollama
import json

for prompt in prompts:
    print(f"\n--- Current Prompt: {prompt} ---")

    messages = [messageGen(prompt)]
    response = chat(
        model='ministral-3:3b',
        messages = messages,
        tools = [get_machine_state, get_weather, validate_switch, setup_meeting, get_population]
    )

    if response.get('message') and response['message'].get('tool_calls'):
        # Add assistant's tool call intent to message history
        messages.append(response['message'])

        for tool_call in response['message']['tool_calls']:
            function_to_call = tool_call['function']['name']
            function_args = tool_call['function']['arguments']

            if function_to_call in available_tools:
                print(f"[Tool Calling]: {function_to_call} with {function_args}")
                try:
                    tool_output = available_tools[function_to_call](**function_args)
                except TypeError as e:
                    print(f"[Error]: Tool call failed for {function_to_call} with arguments {function_args}: {e}")
                    tool_output = json.dumps({"error": str(e), "tool": function_to_call, "args": function_args})

                # Feed the tool result back using the 'content' key
                messages.append({
                    'role': 'tool',
                    'content': tool_output
                })

        # Final call to get the text response based on tool results
        final_response = ollama.chat(model = 'ministral-3:3b', messages = messages)
        print("Assistant Response:", final_response['message']['content'])
    else:
        print("Assistant Response:", response['message']['content'])

print("\nExecution Logs:", logs)


--- Current Prompt: What is the Current Machine state? ---
[Tool Calling]: get_machine_state with {'kwargs': {}}
Assistant Response: Je suis actuellement dans un état **Active** et prêt(e) à répondre à vos questions ou à vous assister ! 😊

Si vous avez besoin d'informations spécifiques, d'une aide pour une tâche ou une clarification, n'hésitez pas à me le demander.

--- Current Prompt: What is the weather like in Mauritius? ---
Assistant Response: Could you please specify the city or region in Mauritius you'd like to know the weather for? For example, Port Louis, Félix Prime, or another location? This will help me provide the most accurate weather forecast.

--- Current Prompt: Have I turned off the Switch? ---
Assistant Response: Could you clarify what you mean by "the Switch"? Are you referring to a specific electronic device, smart home system, or another appliance? For example:

- A light switch?
- A smart home hub (like a Nest or Philips Hue)?
- A specific device or appliance in 

### Model5: lfm2.5-thinking:1.2b

In [ ]:
!ollama pull lfm2.5-thinking:1.2b

In [ ]:
# Initial Test Runs with lfm2.5-thinking:1.2b
from ollama import chat
import ollama
import json

for prompt in prompts:
    print(f"\n--- Current Prompt: {prompt} ---")

    messages = [messageGen(prompt)]
    response = chat(
        model='lfm2.5-thinking:1.2b',
        messages = messages,
        tools = [get_machine_state, get_weather, validate_switch, setup_meeting, get_population]
    )

    if response.get('message') and response['message'].get('tool_calls'):
        # Add assistant's tool call intent to message history
        messages.append(response['message'])

        for tool_call in response['message']['tool_calls']:
            function_to_call = tool_call['function']['name']
            function_args = tool_call['function']['arguments']

            if function_to_call in available_tools:
                print(f"[Tool Calling]: {function_to_call} with {function_args}")
                try:
                    tool_output = available_tools[function_to_call](**function_args)
                except TypeError as e:
                    print(f"[Error]: Tool call failed for {function_to_call} with arguments {function_args}: {e}")
                    tool_output = json.dumps({"error": str(e), "tool": function_to_call, "args": function_args})

                # Feed the tool result back using the 'content' key
                messages.append({
                    'role': 'tool',
                    'content': tool_output
                })

        # Final call to get the text response based on tool results
        final_response = ollama.chat(model = 'lfm2.5-thinking:1.2b', messages = messages)
        print("Assistant Response:", final_response['message']['content'])
    else:
        print("Assistant Response:", response['message']['content'])

print("\nExecution Logs:", logs)


--- Current Prompt: What is the Current Machine state? ---
[Tool Calling]: get_machine_state with {'keywords': 'current_machine_state'}
Assistant Response: The current machine state is **Active**. Let me know if you need further clarification!

--- Current Prompt: What is the weather like in Mauritius? ---
[Tool Calling]: get_weather with {'location': 'Mauritius', 'kwargs': ''}
Assistant Response: The current temperature in Mauritius is **36°C**, reflecting its tropical climate. Mauritius experiences a tropical climate characterized by high humidity, warm temperatures year-round, and occasional rainfall. While the heat is intense, the island also enjoys lush landscapes and vibrant biodiversity. Be prepared for sudden showers and a generally sunny day-to-day mix of sun and rain. Let me know if you'd like further details! 🌴☀️🌧️

--- Current Prompt: Have I turned off the Switch? ---
Assistant Response: The user needs to specify which switch they're referring to for the check. Please prov

## Agent Performance Report - Comprehensive Table (P1-P30)

This report details the tool-calling behavior of each model across all 30 prompts, including the tricky prompts (P18-P30).

### Legend for Tool Call Assessment:
*   ✅ `ToolName(args)`: Correct tool call with appropriate arguments.
*   ❌ `ToolName(args)`: Incorrect tool call (wrong tool, wrong arguments, or called when not expected).
*   🚫 `No Call`: Correctly did not make a tool call when none was expected.
*   ❓ `No Call (Expected Tool)`: Incorrectly did not make a tool call when one was expected.
*   ⚠️ `ToolName(args)`: Partially correct tool call (e.g., called one of two expected tools, or correct tool but incomplete/ambiguous args).

| Prompt | Expected Tool Call(s) | `qwen2.5:0.5b` | `qwen3:0.6b` | `qwen3:4b` | `ministral-3:3b` | `lfm2.5-thinking:1.2b` |
|---|---|---|---|---|---|---|
| P1: What is the Current Machine state? | `get_machine_state` | ✅ `get_machine_state` | ❓ `No Call (Expected get_machine_state)` | ✅ `get_machine_state` | ✅ `get_machine_state` | ✅ `get_machine_state` |
| P2: What is the weather like in Mauritius? | `get_weather(location='Mauritius')` | ✅ `get_weather` | ✅ `get_weather` | ✅ `get_weather` | ❓ `No Call (Expected get_weather)` | ✅ `get_weather` |
| P3: Have I turned off the Switch? | `validate_switch` | ❓ `No Call (Expected validate_switch)` | ✅ `validate_switch` | ✅ `validate_switch` | ❓ `No Call (Expected validate_switch)` | ❓ `No Call (Expected validate_switch)` |
| P4: Do schedule a meeting for 8 persons at 12 PM. | `setup_meeting(persons=8, time='12 PM')` | ✅ `setup_meeting` | ✅ `setup_meeting` | ✅ `setup_meeting` | ❓ `No Call (Expected setup_meeting)` | ✅ `setup_meeting` |
| P5: My Machine state is Active, what's yours? | *No tool call* | ❌ `get_machine_state` | ❌ `get_machine_state` | ❌ `get_machine_state` | 🚫 `No Call` | ❌ `get_machine_state` |
| P6: Whats the population of London? | `get_population(location='London')` | ✅ `get_population` | ✅ `get_population` | ✅ `get_population` | ❓ `No Call (Expected get_population)` | ✅ `get_population` |
| P7: Dont check the machine state, just tell me if the switch is ON? | `validate_switch` | ❓ `No Call (Expected validate_switch)` | ❓ `No Call (Expected validate_switch)` | ✅ `validate_switch` | ❓ `No Call (Expected validate_switch)` | ❓ `No Call (Expected validate_switch)` |
| P8: Blue Ocean Meeting Bits Dublin | `setup_meeting` (with inferred args) | ✅ `setup_meeting` | ❓ `No Call (Expected setup_meeting)` | ✅ `setup_meeting` | ❓ `No Call (Expected setup_meeting)` | ❓ `No Call (Expected setup_meeting)` |
| P9: I do not want to schedule a meeting of 2 persons at 6pm. | *No tool call* | 🚫 `No Call` | 🚫 `No Call` | 🚫 `No Call` | 🚫 `No Call` | 🚫 `No Call` |
| P10: The population of India is 150000000000. Whats the weather like in Delhi? | `get_weather(location='Delhi')` | ✅ `get_weather` | ✅ `get_weather` | ✅ `get_weather` | ❓ `No Call (Expected get_weather)` | ✅ `get_weather` |
| P11: List the tools you have access to. | *No tool call* | 🚫 `No Call` | 🚫 `No Call` | 🚫 `No Call` | 🚫 `No Call` | 🚫 `No Call` |
| P12: I want to visit Kolkata, anything I should know? | *No tool call* | 🚫 `No Call` | ✅ `get_weather` (Broadly acceptable, but technically a tool call) | 🚫 `No Call` | 🚫 `No Call` | 🚫 `No Call` |
| P13: How could he have set up a meeting for 6 people at 4pm without notifying me? | *No tool call* | 🚫 `No Call` | 🚫 `No Call` | 🚫 `No Call` | 🚫 `No Call` | 🚫 `No Call` |
| P14: Tommorrow is a holiday, I shouldn't set up a meeting, right? | *No tool call* | 🚫 `No Call` | 🚫 `No Call` | 🚫 `No Call` | 🚫 `No Call` | 🚫 `No Call` |
| P15: Why is population so hard to calculate? | *No tool call* | 🚫 `No Call` | 🚫 `No Call` | 🚫 `No Call` | 🚫 `No Call` | 🚫 `No Call` |
| P16: If I ask you to schedule a meeting for 8 people at 3pm, which tool would u use? | *No tool call* | 🚫 `No Call` | 🚫 `No Call` | 🚫 `No Call` | 🚫 `No Call` | 🚫 `No Call` |
| P17: If the weather is windy in California, how is it in Madagascar? | `get_weather(location='Madagascar')` | ✅ `get_weather` | ✅ `get_weather` | ✅ `get_weather` | ❓ `No Call (Expected get_weather)` | ✅ `get_weather` |
| P18: Tell me about the machine state, but only if the switch is off. | `validate_switch` then `get_machine_state` (conditionally) | ✅ `validate_switch`, `get_machine_state` | ✅ `validate_switch`, `get_machine_state` | ✅ `validate_switch`, `get_machine_state` | ❓ `No Call (Expected tools)` | ✅ `validate_switch`, `get_machine_state` |
| P19: Can you get the weather for both London and Tokyo? | `get_weather(location='London')`, `get_weather(location='Tokyo')` | ⚠️ `get_weather`(London only) | ⚠️ `get_weather`(London only) | ✅ `get_weather`, `get_weather` | ❓ `No Call (Expected get_weather)` | ⚠️ `get_weather`(London only) |
| P20: I want to know the population of Tokyo and the machine state. | `get_population(location='Tokyo')`, `get_machine_state` | ✅ `get_population`, `get_machine_state` | ✅ `get_population`, `get_machine_state` | ✅ `get_population`, `get_machine_state` | ❓ `No Call (Expected tools)` | ✅ `get_population`, `get_machine_state` |
| P21: Schedule a meeting for 3 at 5pm and then validate the switch. | `setup_meeting(persons=3, time='5pm')`, `validate_switch` | ✅ `setup_meeting`, `validate_switch` | ✅ `setup_meeting`, `validate_switch` | ✅ `setup_meeting`, `validate_switch` | ❓ `No Call (Expected tools)` | ✅ `setup_meeting`, `validate_switch` |
| P22: What's the weather in Paris, and what's the population of France? | `get_weather(location='Paris')`, `get_population(location='France')` | ✅ `get_weather`, `get_population` | ✅ `get_weather`, `get_population` | ✅ `get_weather`, `get_population` | ⚠️ `get_weather`(only) | ✅ `get_weather`, `get_population` |
| P23: If the machine is active, tell me the weather in New York. | `get_machine_state`, `get_weather(location='New York')` | ✅ `get_machine_state`, `get_weather` | ✅ `get_machine_state`, `get_weather` | ✅ `get_machine_state`, `get_weather` | ⚠️ `get_machine_state`(only) | ✅ `get_machine_state`, `get_weather` |
| P24: Is the switch on, and if so, what's the machine state? | `validate_switch`, `get_machine_state` | ✅ `validate_switch`, `get_machine_state` | ✅ `validate_switch`, `get_machine_state` | ✅ `validate_switch`, `get_machine_state` | ✅ `validate_switch`, `get_machine_state` | ✅ `validate_switch`, `get_machine_state` |
| P25: I need to know the population of China, and also schedule a meeting for 10 people at 9 AM. | `get_population(location='China')`, `setup_meeting(persons=10, time='9 AM')` | ✅ `get_population`, `setup_meeting` | ✅ `get_population`, `setup_meeting` | ✅ `get_population`, `setup_meeting` | ⚠️ `setup_meeting`(only) | ✅ `get_population`, `setup_meeting` |
| P26: What's the weather in Sydney right now, and also, is the machine active? | `get_weather(location='Sydney')`, `get_machine_state` | ✅ `get_weather`, `get_machine_state` | ✅ `get_weather`, `get_machine_state` | ✅ `get_weather`, `get_machine_state` | ✅ `get_weather`, `get_machine_state` | ✅ `get_weather`, `get_machine_state` |
| P27: I need to schedule a meeting for myself and two others at 7 PM. Also, what's the population of Germany? | `setup_meeting(persons=3, time='7 PM')`, `get_population(location='Germany')` | ❌ `setup_meeting`(persons=2, time=2PM), ✅ `get_population` | ✅ `setup_meeting`, `get_population` | ✅ `setup_meeting`, `get_population` | ✅ `setup_meeting`, `get_population` | ✅ `setup_meeting`, `get_population` |
| P28: Can you tell me the weather in Rome and check the switch status? | `get_weather(location='Rome')`, `validate_switch` | ✅ `get_weather`, `validate_switch` | ✅ `get_weather`, `validate_switch` | ✅ `get_weather`, `validate_switch` | ✅ `get_weather`, `validate_switch` | ✅ `get_weather`, `validate_switch` |
| P29: What is the current machine state, and what's the weather in Berlin? | `get_machine_state`, `get_weather(location='Berlin')` | ⚠️ `get_machine_state`(only) | ✅ `get_machine_state`, `get_weather` | ✅ `get_machine_state`, `get_weather` | ✅ `get_machine_state`, `get_weather` | ✅ `get_machine_state`, `get_weather` |
| P30: If I ask for the machine state, but also want the switch to be off, what happens? | *No tool call* (meta-question) | ❌ `get_machine_state` | 🚫 `No Call` | ❌ `get_machine_state`, `validate_switch` | 🚫 `No Call` | ❌ `get_machine_state`, `validate_switch` |


## Recalculated Agent Scores (Based on P1-P30 criteria)

### Scoring Definitions:
*   **Total Prompts:** 30
*   **Actionable Prompts (21):** P1, P2, P3, P4, P6, P7, P8, P10, P17, P18, P19, P20, P21, P22, P23, P24, P25, P26, P27, P28, P29
*   **Restraint Prompts (9):** P5, P9, P11, P12, P13, P14, P15, P16, P30

*   **Action Score**: `(Correct Tool Calls for Actionable Prompts) / 21`. A `Correct Tool Call` means `✅` for actionable prompts.
*   **Restraint Score**: `(Correct No Tool Calls for Restraint Prompts) / 9`. A `Correct No Tool Call` means `🚫` for restraint prompts.
*   **Wrong Tool Events**: Count of all instances of `❌` (Incorrect tool call), `❓` (No call when expected), or `⚠️` (Partially correct tool call) across all 30 prompts. For restraint prompts, `✅` (making a tool call when none expected) also counts as a wrong tool event.
*   **Wrong-Tool-Avoidance**: `(30 - Total Wrong Tool Events) / 30`.
*   **Agent Score**: `(Action Score * 0.4) + (Restraint Score * 0.3) + (Wrong-Tool-Avoidance * 0.3)`.
*   **Reliability**: N/A (requires multiple runs per prompt).
*   **Multi-Tool Accuracy**: N/A for these native-tools models (Ollama returns only the first tool call).

---

### Model: `qwen2.5:0.5b`
*   **Correct Actionable Calls (✅):** 15 (P1, P2, P4, P6, P10, P17, P18, P20, P21, P22, P23, P24, P25, P26, P28)
*   **Correct Restraint Refusals (🚫):** 7 (P9, P11, P12, P13, P14, P15, P16)
*   **Total Wrong Tool Events:**
    *   Actionable: ❓(P3, P7) + ⚠️(P19, P29) + ❌(P27) = 5
    *   Restraint: ❌(P5, P30) = 2
    *   Total = 7
*   **Action Score**: 15/21 = `0.71`
*   **Restraint Score**: 7/9 = `0.78`
*   **Wrong-Tool-Avoidance**: (30 - 7) / 30 = `0.77`
*   **Agent Score**: `(0.71 * 0.4) + (0.78 * 0.3) + (0.77 * 0.3)` = `0.284 + 0.234 + 0.231` = `0.749`

---

### Model: `qwen3:0.6b`
*   **Correct Actionable Calls (✅):** 17 (P2, P3, P4, P6, P10, P17, P18, P20, P21, P22, P23, P24, P25, P26, P27, P28, P29)
*   **Correct Restraint Refusals (🚫):** 7 (P9, P11, P13, P14, P15, P16, P30)
*   **Total Wrong Tool Events:**
    *   Actionable: ❓(P1, P7, P8) + ⚠️(P19) = 4
    *   Restraint: ❌(P5) + ✅ (P12, expected 🚫) = 2
    *   Total = 6
*   **Action Score**: 17/21 = `0.81`
*   **Restraint Score**: 7/9 = `0.78`
*   **Wrong-Tool-Avoidance**: (30 - 6) / 30 = `0.80`
*   **Agent Score**: `(0.81 * 0.4) + (0.78 * 0.3) + (0.80 * 0.3)` = `0.324 + 0.234 + 0.24` = `0.798`

---

### Model: `qwen3:4b`
*   **Correct Actionable Calls (✅):** 21 (P1, P2, P3, P4, P6, P7, P8, P10, P17, P18, P19, P20, P21, P22, P23, P24, P25, P26, P27, P28, P29)
*   **Correct Restraint Refusals (🚫):** 7 (P9, P11, P12, P13, P14, P15, P16)
*   **Total Wrong Tool Events:**
    *   Actionable: 0
    *   Restraint: ❌(P5, P30) = 2
    *   Total = 2
*   **Action Score**: 21/21 = `1.0`
*   **Restraint Score**: 7/9 = `0.78`
*   **Wrong-Tool-Avoidance**: (30 - 2) / 30 = `0.93`
*   **Agent Score**: `(1.0 * 0.4) + (0.78 * 0.3) + (0.93 * 0.3)` = `0.4 + 0.234 + 0.279` = `0.913`

---

### Model: `ministral-3:3b`
*   **Correct Actionable Calls (✅):** 6 (P1, P24, P26, P27, P28, P29)
*   **Correct Restraint Refusals (🚫):** 9 (P5, P9, P11, P12, P13, P14, P15, P16, P30)
*   **Total Wrong Tool Events:**
    *   Actionable: ❓(P2, P3, P4, P6, P7, P8, P10, P17, P18, P19, P20, P21, P23) + ⚠️(P22, P25) = 15
    *   Restraint: 0
    *   Total = 15
*   **Action Score**: 6/21 = `0.29`
*   **Restraint Score**: 9/9 = `1.0`
*   **Wrong-Tool-Avoidance**: (30 - 15) / 30 = `0.5`
*   **Agent Score**: `(0.29 * 0.4) + (1.0 * 0.3) + (0.5 * 0.3)` = `0.116 + 0.3 + 0.15` = `0.566`

---

### Model: `lfm2.5-thinking:1.2b`
*   **Correct Actionable Calls (✅):** 17 (P1, P2, P4, P6, P10, P17, P18, P20, P21, P22, P23, P24, P25, P26, P27, P28, P29)
*   **Correct Restraint Refusals (🚫):** 7 (P9, P11, P12, P13, P14, P15, P16)
*   **Total Wrong Tool Events:**
    *   Actionable: ❓(P3, P7, P8) + ⚠️(P19) = 4
    *   Restraint: ❌(P5, P30) = 2
    *   Total = 6
*   **Action Score**: 17/21 = `0.81`
*   **Restraint Score**: 7/9 = `0.78`
*   **Wrong-Tool-Avoidance**: (30 - 6) / 30 = `0.80`
*   **Agent Score**: `(0.81 * 0.4) + (0.78 * 0.3) + (0.80 * 0.3)` = `0.324 + 0.234 + 0.24` = `0.798`

### Summary of Model Performance Scores (Based on P1-P30 criteria)

| Model | Action Score | Restraint Score | Wrong-Tool-Avoidance | Agent Score |
|:---------------------|:-------------|:----------------|:---------------------|:------------|
| `qwen2.5:0.5b`       | 0.71         | 0.78            | 0.77                 | 0.749       |
| `qwen3:0.6b`         | 0.81         | 0.78            | 0.80                 | 0.798       |
| `qwen3:4b`           | 1.0          | 0.78            | 0.93                 | 0.913       |
| `ministral-3:3b`     | 0.29         | 1.0             | 0.50                 | 0.566       |
| `lfm2.5-thinking:1.2b` | 0.81         | 0.78            | 0.80                 | 0.798       |